# Notebook 8 — Ablations: state effects + RSF capacity in the national model

Three matched experiments on the Notebook 4 design matrix:

1. **Do the 51 `STATE_*` dummies help or hurt?** (arms A/B) Geography already enters
   through the continuous climate and coastal features, so state identity may be
   redundant — or may even hurt within-state ranking by letting trees split on the
   state code instead of physical covariates.
2. **Does a dedicated single-state model beat the pooled model within its state?**
   (arm C) Pooled vs per-state C-index score different tasks — the pooled C-index earns
   credit for between-state comparisons; a per-state C-index only scores within-state
   ranking, which is harder (national numbers: `us_rsf_metrics.json` +
   `us_rsf_cindex_by_state.csv`). Arm C asks how much of MA's gap a specialist trained
   only on MA recovers, holding features and test rows fixed.
3. **Is the full RSF's gap to Cox/AFT a real capacity limit or a coarsening artifact?**
   (arm D) The full model is coarsened to fit in RAM (leaf=50, 10% per-tree
   subsampling). Arm D reruns the RSF at MA-grade hyperparameters (leaf=5, split=10,
   300 trees, full bootstrap) on **cost-matched national subsamples** (sized to Arm C's
   training set, 5 seeds) plus a **capacity curve** at 50k/100k/200k rows, all scored on
   the full shared test set — the affordable way to bound how much C-index the coarsening
   costs and whether more data would close the gap.

**Arms** (A/B share one stratified 300k training subsample and the full shared test set;
C trains and evaluates on the MA subset of the same split; D trains on small national
subsamples and evaluates on the full shared test set):

| Arm | Training rows | Features | Question it answers |
|---|---|---|---|
| A | 300k national | current set (STATE_* kept) | baseline at matched cost |
| B | 300k national | STATE_* dropped | does removing state code help within-state ranking? |
| C | MA only (~13k) | national set, no STATE_* | does an MA specialist beat the pooled model on MA? |
| D | 15k cost-matched + 50k/100k/200k curve, MA-grade params | full national set | how much of the RSF-vs-parametric gap is coarsening, and does more data close it? |

C uses the national matrix's features, so it lacks the state-DOT maintenance features a
purpose-built MA model could add (`../ma_bridges/`).

**Decision rule (A/B):** drop STATE_* from the full model (`DROP_STATE_DUMMIES = True` in
`train_national_rsf.ipynb`) only if B beats A on median per-state C-index without losing
pooled C-index beyond noise (±0.005).

**Run order:** requires Notebook 4's matrix plus `us_rsf_metrics.json` (Notebook 5) and
`us_parametric_metrics.json` (Notebook 6) for Arm D's reference numbers.

In [1]:
# ── Configuration + typed load (same load pattern as train_national_rsf.ipynb) ──
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored

DATA_CLEAN   = Path("us_rsf_data_clean.csv")
RSF_METRICS  = Path("us_rsf_metrics.json")          # Arm D reference (Notebook 5)
PARA_METRICS = Path("us_parametric_metrics.json")   # Arm D reference (Notebook 6)
OUT_JSON     = Path("us_ablation_state_dummies.json")
KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]

SMOKE_TEST = False   # True -> run on the Notebook 4 _smoke matrix with tiny forests

ABLATION_SAMPLE = 300_000
# Matched-cost arms: ~1/3 the full Notebook 5 workload, enough for stable comparisons.
# low_memory=True: only risk scores are needed here (no survival curves).
ABLATION_PARAMS = dict(n_estimators=100, max_samples=0.2, min_samples_split=100,
                       min_samples_leaf=50, max_features="sqrt", low_memory=True,
                       n_jobs=-1, random_state=42)
# MA-standalone capacity (../ma_bridges/RSF.ipynb, leaf=5, split=10); 300 trees match its
# C-index at a third of the cost. Arm D reuses these on cost-matched national subsamples.
MA_PARAMS = dict(n_estimators=300, min_samples_split=10, min_samples_leaf=5,
                 max_features="sqrt", low_memory=True, n_jobs=-1, random_state=42)

if SMOKE_TEST:
    if Path("us_rsf_data_clean_smoke.csv").exists():
        DATA_CLEAN   = Path("us_rsf_data_clean_smoke.csv")
        RSF_METRICS  = Path("us_rsf_metrics_smoke.json")
        PARA_METRICS = Path("us_parametric_metrics_smoke.json")
    OUT_JSON = OUT_JSON.with_stem(OUT_JSON.stem + "_smoke")
    ABLATION_SAMPLE = 5_000
    ABLATION_PARAMS.update(n_estimators=25)
    MA_PARAMS.update(n_estimators=50)
    print(f"SMOKE_TEST: {DATA_CLEAN}, tiny forests, outputs -> {OUT_JSON}")

cols = pd.read_csv(DATA_CLEAN, nrows=0).columns
dtypes = {c: "float32" for c in cols}
dtypes.update({k: str for k in KEYS})
dtypes["event"] = "int8"
df = pd.read_csv(DATA_CLEAN, dtype=dtypes)
for k in KEYS:
    df[k] = df[k].str.strip()

y   = Surv.from_arrays(event=df["event"].astype(bool).to_numpy(),
                       time=df["time"].astype("float64").to_numpy())
ids = df[KEYS].copy()
X = df.drop(columns=["event", "time", "bridge_age"] + KEYS)   # bridge_age == time; never a feature
del df

STATE_COLS = [c for c in X.columns if c.startswith("STATE_")]
assert STATE_COLS, "no STATE_* dummies in the matrix — nothing to ablate"
assert "STRUCTURE_TYPE_043B_19" not in X.columns, \
    "culvert dummy present — the matrix predates the Notebook 4 culvert exclusion; rebuild it"
print(f"X: {X.shape[0]:,} x {X.shape[1]}   STATE_* dummies: {len(STATE_COLS)}")

X: 973,905 x 255   STATE_* dummies: 51


In [2]:
# ── Shared split + stratified ablation subsample ──────────────────────────────
# Same 80/20 split as Notebook 5 (random_state=42) -> every arm shares one test set.
X_train, X_test, y_train, y_test, ids_train, ids_test = \
    train_test_split(X, y, ids, test_size=0.2, random_state=42)

# Stratified (state, event) subsample keeps each state's event mix intact at 300k rows.
strat = pd.DataFrame({"state": ids_train["STATE_FIPS"].to_numpy(), "event": y_train["event"]})
pos = (strat.groupby(["state", "event"], group_keys=False)
            .sample(frac=min(1.0, ABLATION_SAMPLE / len(strat)), random_state=42)).index.to_numpy()
X_sub, y_sub = X_train.iloc[pos], y_train[pos]
print(f"ablation train: {len(X_sub):,}   shared test: {len(X_test):,}   "
      f"test event rate: {y_test['event'].mean()*100:.1f}%")

ablation train: 299,998   shared test: 194,781   test event rate: 29.3%


In [3]:
# ── Arms A (with STATE_*) and B (without) ─────────────────────────────────────
def cindex_by_state(risk):
    """Per-state C-index with Notebook 5's stability filter (n>=200, events>=30)."""
    rows, state_arr = [], ids_test["STATE_FIPS"].to_numpy()
    for st in np.unique(state_arr):
        m = state_arr == st
        n, n_ev = int(m.sum()), int(y_test["event"][m].sum())
        if n < 200 or n_ev < 30:
            continue
        ci = concordance_index_censored(y_test["event"][m], y_test["time"][m], risk[m])[0]
        rows.append({"STATE_FIPS": st, "n_test": n, "n_events": n_ev, "c_index": ci})
    return pd.DataFrame(rows)

def run_arm(name, Xtr, ytr, Xte, params):
    t0 = time.time()
    rsf = RandomSurvivalForest(**params)
    rsf.fit(Xtr, ytr)
    risk = np.concatenate([rsf.predict(Xte.iloc[i:i+50_000])
                           for i in range(0, len(Xte), 50_000)])
    print(f"[{name}] {Xtr.shape[1]} features, fit+predict {(time.time()-t0)/60:.1f} min", flush=True)
    return risk

risk_A = run_arm("A: with STATE_*",    X_sub, y_sub, X_test, ABLATION_PARAMS)
risk_B = run_arm("B: without STATE_*", X_sub.drop(columns=STATE_COLS), y_sub,
                 X_test.drop(columns=STATE_COLS), ABLATION_PARAMS)

pooled, per_state = {}, {}
for name, risk in [("A", risk_A), ("B", risk_B)]:
    pooled[name] = concordance_index_censored(y_test["event"], y_test["time"], risk)[0]
    per_state[name] = cindex_by_state(risk)
    print(f"arm {name}: pooled C-index {pooled[name]:.4f}")

cmp = per_state["A"].merge(per_state["B"], on=["STATE_FIPS", "n_test", "n_events"],
                           suffixes=("_A", "_B"))
cmp["delta_B_minus_A"] = cmp["c_index_B"] - cmp["c_index_A"]
ma_row = cmp[cmp["STATE_FIPS"] == "25"]
print(f"\nper-state median: A {cmp['c_index_A'].median():.4f}   B {cmp['c_index_B'].median():.4f}")
print(f"median delta (B-A): {cmp['delta_B_minus_A'].median():+.4f}   "
      f"states better under B: {int((cmp['delta_B_minus_A'] > 0).sum())}/{len(cmp)}")
if len(ma_row):
    print(f"MA: A {ma_row['c_index_A'].iloc[0]:.4f} -> B {ma_row['c_index_B'].iloc[0]:.4f}")
cmp.sort_values("delta_B_minus_A")

[A: with STATE_*] 255 features, fit+predict 5.6 min


[B: without STATE_*] 204 features, fit+predict 6.3 min


arm A: pooled C-index 0.7542


arm B: pooled C-index 0.7532

per-state median: A 0.7457   B 0.7472
median delta (B-A): +0.0019   states better under B: 34/50
MA: A 0.6755 -> B 0.6715


,STATE_FIPS,n_test,n_events,c_index_A,c_index_B,delta_B_minus_A
24,29,12289,3049,0.616789,0.603718,-0.013071
34,39,8197,2673,0.758309,0.748386,-0.009924
12,17,15019,3305,0.681327,0.674982,-0.006345
4,06,5484,1842,0.734761,0.729610,-0.005151
20,25,3836,813,0.675479,0.671527,-0.003952
48,55,3616,1232,0.726038,0.722114,-0.003924
19,24,1002,278,0.840361,0.836818,-0.003543
32,37,9515,2429,0.744439,0.742222,-0.002217
45,51,4967,946,0.749925,0.747909,-0.002015
30,35,971,238,0.739817,0.738209,-0.001608


In [4]:
# ── Arm C: MA-only specialist benchmark ───────────────────────────────────────────────
# Same split membership, so C evaluates on exactly the MA rows of the shared test set —
# directly comparable to the national model's within-MA C-index (FIPS 25 row of
# us_rsf_cindex_by_state.csv, Notebook 5).
ma_tr = ids_train["STATE_FIPS"].to_numpy() == "25"
ma_te = ids_test["STATE_FIPS"].to_numpy() == "25"
Xtr_ma = X_train[ma_tr].drop(columns=STATE_COLS)     # constant within one state
Xte_ma = X_test[ma_te].drop(columns=STATE_COLS)
ytr_ma, yte_ma = y_train[ma_tr], y_test[ma_te]
print(f"MA train: {len(Xtr_ma):,}   MA test: {len(Xte_ma):,}")

t0 = time.time()
rsf_C = RandomSurvivalForest(**MA_PARAMS).fit(Xtr_ma, ytr_ma)
ci_C = concordance_index_censored(yte_ma["event"], yte_ma["time"], rsf_C.predict(Xte_ma))[0]
print(f"C (MA-only specialist): {ci_C:.4f}   ({(time.time()-t0)/60:.1f} min)", flush=True)

MA train: 15,057   MA test: 3,836


C (MA-only specialist): 0.8356   (0.1 min)


In [5]:
# ── Arm D: cost-matched capacity check (is the RSF gap real or a coarsening artifact?) ──
# Bounds how much of the full RSF's gap to Cox/AFT is genuine capacity vs the memory-driven
# coarsening (leaf=50, 10% subsampling): MA-grade hyperparameters (Arm C's MA_PARAMS) on
# national subsamples sized to the MA specialist's training set, so per-run cost matches
# Arm C. 5 seeds vary subsample and forest; each scores the full shared test set. A
# capacity curve (50k/100k/200k) then tracks how the RSF closes on Cox as data grows
rsf_ref  = json.loads(RSF_METRICS.read_text())
para_ref = json.loads(PARA_METRICS.read_text())

n_match = int(ma_tr.sum())          # MA-specialist training size
D_SEEDS = 2 if SMOKE_TEST else 5
# (target rows, seeds) for the capacity curve; single-seed points anchor on seed 42
CURVE = [(10_000, 1), (20_000, 1)] if SMOKE_TEST else [(50_000, 1), (100_000, 3), (200_000, 1)]
strat_D = pd.DataFrame({"state": ids_train["STATE_FIPS"].to_numpy(), "event": y_train["event"]})

def run_arm_d(n_rows, seed):
    pos = (strat_D.groupby(["state", "event"], group_keys=False)
                  .sample(frac=min(1.0, n_rows / len(strat_D)), random_state=seed)
           ).index.to_numpy()
    t0 = time.time()
    rsf_D = RandomSurvivalForest(**{**MA_PARAMS, "random_state": seed})
    rsf_D.fit(X_train.iloc[pos], y_train[pos])
    risk = np.concatenate([rsf_D.predict(X_test.iloc[i:i+50_000])
                           for i in range(0, len(X_test), 50_000)])
    ci_pool = concordance_index_censored(y_test["event"], y_test["time"], risk)[0]
    per_st = cindex_by_state(risk)
    ci_med = float(per_st["c_index"].median()) if len(per_st) else float("nan")
    print(f"  seed {seed}: n_train {len(pos):,}  pooled C {ci_pool:.4f}  "
          f"median per-state C {ci_med:.4f}  ({(time.time()-t0)/60:.1f} min)", flush=True)
    return {"seed": seed, "n_train": int(len(pos)),
            "pooled_c_index": round(float(ci_pool), 4),
            "median_state_c_index": round(ci_med, 4)}

print(f"Arm D: MA-grade params on {n_match:,}-row national subsamples, {D_SEEDS} seeds")
arm_D_runs = [run_arm_d(n_match, seed) for seed in range(D_SEEDS)]

pooled_D = np.array([r["pooled_c_index"] for r in arm_D_runs])
median_D = np.array([r["median_state_c_index"] for r in arm_D_runs])
print(f"\nArm D pooled C: {pooled_D.mean():.4f} ± {pooled_D.std(ddof=1):.4f}   "
      f"median per-state C: {median_D.mean():.4f} ± {median_D.std(ddof=1):.4f}")

print("\nCapacity curve: MA-grade params at growing training size "
      "(fine params can't reach the full 779k — memory-bound past ~200k)")
capacity_curve = []
for n_rows, k in CURVE:
    seeds = [42] + list(range(k - 1))          # seed 42 anchors every point
    runs = [run_arm_d(n_rows, s) for s in seeds]
    cs = np.array([r["pooled_c_index"] for r in runs])
    ms = np.array([r["median_state_c_index"] for r in runs])
    capacity_curve.append({
        "n_train_target": n_rows, "n_seeds": k,
        "pooled_c_mean": round(float(cs.mean()), 4),
        "pooled_c_sd": round(float(cs.std(ddof=1)), 4) if k > 1 else None,
        "median_state_c_mean": round(float(ms.mean()), 4),
        "per_seed": runs,
    })

full_rsf_c = rsf_ref["c_index_test"]
cox_c, aft_c = para_ref["cox_ph"]["c_index_test"], para_ref["weibull_aft"]["c_index_test"]
gap_full = cox_c - full_rsf_c
gap_D    = cox_c - pooled_D.mean()
top = capacity_curve[-1]                        # largest-n capacity point
gap_top = cox_c - top["pooled_c_mean"]

print(f"\n{'model / arm':<50} {'pooled C':>9}")
print(f"{'full RSF (coarsened: leaf=50, 10% subsample, 779k)':<50} {full_rsf_c:>9.4f}")
print(f"{f'Arm D  (MA-grade, {n_match:,} rows, mean of {D_SEEDS})':<50} {pooled_D.mean():>9.4f}")
for pt in capacity_curve:
    lbl = f"  capacity (MA-grade, {pt['n_train_target']:,} rows, x{pt['n_seeds']})"
    sd = f" ± {pt['pooled_c_sd']:.4f}" if pt['pooled_c_sd'] is not None else ""
    print(f"{lbl:<50} {pt['pooled_c_mean']:>9.4f}{sd}")
print(f"{'Cox PH (full scale)':<50} {cox_c:>9.4f}")
print(f"{'Weibull AFT (full scale)':<50} {aft_c:>9.4f}")
print(f"\nRSF-vs-Cox gap: {gap_full:+.4f} at full coarsened scale, {gap_D:+.4f} at the "
      f"{n_match:,}-row cost-matched point, {gap_top:+.4f} at the {top['n_train_target']:,}-row "
      f"capacity ceiling\n-> "
      f"{'most of the gap survives de-coarsening (capacity is real)' if gap_top > 0.5 * gap_full else 'much of the gap was a coarsening artifact'}")

Arm D: MA-grade params on 15,057-row national subsamples, 5 seeds


  seed 0: n_train 15,056  pooled C 0.7875  median per-state C 0.7742  (0.8 min)


  seed 1: n_train 15,056  pooled C 0.7885  median per-state C 0.7804  (0.8 min)


  seed 2: n_train 15,056  pooled C 0.7908  median per-state C 0.7758  (0.8 min)


  seed 3: n_train 15,056  pooled C 0.7885  median per-state C 0.7826  (0.8 min)


...

  seed 42: n_train 99,999  pooled C 0.8016  median per-state C 0.7919  (11.0 min)


  seed 0: n_train 99,999  pooled C 0.7991  median per-state C 0.7950  (12.1 min)


  seed 1: n_train 99,999  pooled C 0.8043  median per-state C 0.7898  (12.0 min)


  seed 42: n_train 199,998  pooled C 0.8063  median per-state C 0.7978  (75.6 min)



model / arm                                         pooled C
full RSF (coarsened: leaf=50, 10% subsample, 779k)    0.7526
Arm D  (MA-grade, 15,057 rows, mean of 5)             0.7872
  capacity (MA-grade, 50,000 rows, x1)                0.7963
  capacity (MA-grade, 100,000 rows, x3)               0.8017 ± 0.0026
  capacity (MA-grade, 200,000 rows, x1)               0.8063
Cox PH (full scale)                                   0.8830
Weibull AFT (full scale)                              0.8833

RSF-vs-Cox gap: +0.1304 at full coarsened scale, +0.0958 at the 15,057-row cost-matched point, +0.0767 at the 200,000-row capacity ceiling
-> most of the gap survives de-coarsening (capacity is real)


In [6]:
# ── Summary -> us_ablation_state_dummies.json ─────────────────────────────────
results = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "design": {
        "ablation_train_rows": int(len(X_sub)),
        "shared_test_rows": int(len(X_test)),
        "ablation_params": {k: (v if isinstance(v, (int, float, str, bool, type(None)))
                                else str(v)) for k, v in ABLATION_PARAMS.items()},
        "ma_params": {k: (v if isinstance(v, (int, float, str, bool, type(None)))
                          else str(v)) for k, v in MA_PARAMS.items()},
        "n_state_dummies": len(STATE_COLS),
    },
    "arm_A_with_state": {
        "pooled_c_index": round(float(pooled["A"]), 4),
        "median_state_c_index": round(float(cmp["c_index_A"].median()), 4),
        "ma_c_index": round(float(ma_row["c_index_A"].iloc[0]), 4) if len(ma_row) else None,
    },
    "arm_B_without_state": {
        "pooled_c_index": round(float(pooled["B"]), 4),
        "median_state_c_index": round(float(cmp["c_index_B"].median()), 4),
        "ma_c_index": round(float(ma_row["c_index_B"].iloc[0]), 4) if len(ma_row) else None,
    },
    "delta_summary": {
        "median_state_delta_B_minus_A": round(float(cmp["delta_B_minus_A"].median()), 4),
        "states_better_under_B": int((cmp["delta_B_minus_A"] > 0).sum()),
        "n_states_compared": int(len(cmp)),
    },
    "arm_C_ma_specialist_c_index": round(float(ci_C), 4),
    "arm_D_cost_matched_capacity": {
        "question": ("How much of the full RSF's C-index gap to Cox/AFT is the memory-driven "
                     "coarsening (leaf=50, 10% per-tree subsampling) vs real capacity limits? "
                     "MA-grade hyperparameters at Arm-C-matched cost on national subsamples."),
        "params": {k: (v if isinstance(v, (int, float, str, bool, type(None)))
                       else str(v)) for k, v in MA_PARAMS.items()},
        "n_train_per_seed": n_match,
        "n_seeds": D_SEEDS,
        "pooled_c_mean": round(float(pooled_D.mean()), 4),
        "pooled_c_sd": round(float(pooled_D.std(ddof=1)), 4),
        "median_state_c_mean": round(float(median_D.mean()), 4),
        "median_state_c_sd": round(float(median_D.std(ddof=1)), 4),
        "per_seed": arm_D_runs,
        "capacity_curve": capacity_curve,
        "full_rsf_reference": full_rsf_c,
        "cox_reference": cox_c,
        "aft_reference": aft_c,
        "gap_to_cox_full_rsf": round(float(gap_full), 4),
        "gap_to_cox_arm_D": round(float(gap_D), 4),
        "gap_to_cox_top_capacity": round(float(gap_top), 4),
    },
}
OUT_JSON.write_text(json.dumps(results, indent=2))
print(f"saved {OUT_JSON}\n")
print(json.dumps(results, indent=2))

saved us_ablation_state_dummies.json

{
  "generated_utc": "2026-07-20T08:41:18Z",
  "design": {
    "ablation_train_rows": 299998,
    "shared_test_rows": 194781,
    "ablation_params": {
      "n_estimators": 100,
      "max_samples": 0.2,
      "min_samples_split": 100,
      "min_samples_leaf": 50,
      "max_features": "sqrt",
      "low_memory": true,
      "n_jobs": -1,
      "random_state": 42
    },
    "ma_params": {
      "n_estimators": 300,
      "min_samples_split": 10,
      "min_samples_leaf": 5,
      "max_features": "sqrt",
      "low_memory": true,
      "n_jobs": -1,
      "random_state": 42
    },
    "n_state_dummies": 51
  },
  "arm_A_with_state": {
    "pooled_c_index": 0.7542,
    "median_state_c_index": 0.7457,
    "ma_c_index": 0.6755
  },
  "arm_B_without_state": {
    "pooled_c_index": 0.7532,
    "median_state_c_index": 0.7472,
    "ma_c_index": 0.6715
  },
  "delta_summary": {
    "median_state_delta_B_minus_A": 0.0019,
    "states_better_under_B": 34,
  